# Practicum 1 — dynamic bank closure

This notebook estimates monetary closure cost, cross-fitted closure CCPs,
conditional continuation values, and the structural parameters requested by
the submission template.

A/B use calibrated beta=0.96. C uses beta=0.70. Beta is displayed for
transparency but is not a submission row.

In [2]:
from pathlib import Path

import pandas as pd

from dse_practicum import (
    P1CounterfactualConfig,
    P1FitConfig,
    fit_practicum1,
    fit_practicum1_all,
)

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "practicum" / "practicum1" / "data").exists()
)

DATA = ROOT / "practicum" / "practicum1" / "data"
DATA

PosixPath('/Users/bc3139/repo/summer2026/DSE2026/practicum/practicum1/data')

## 1. Fit one dataset first

This is the recommended debugging workflow. The full settings use three CCP
folds and 200 boosting iterations.

In [3]:
config = P1FitConfig()

result_a = fit_practicum1(
    DATA / "set_A.parquet",
    spec="A",
    config=config,
)
result_a.summary()

,estimate
quantity,
sigma,47.813325
beta (calibration; not submitted),0.960000
theta_const,-356.161916
theta_log_assets,459.366358
theta_log_assets^2,-20.268342
theta_npf_a,6821.286014
theta_roa,-829.263120
theta_realest_a,4516.696201
theta_house,0.038824


In [4]:
pd.Series(result_a.diagnostics, name="value").to_frame()

,value
n_rows,154068.000000
n_banks,8000.000000
n_failures,4510.000000
monetary_cost_rmse,200.176235
monetary_cost_r_squared,0.681820
ccp_event_rate,0.029273
ccp_log_loss,0.128571
ccp_brier_score,0.028230
ccp_roc_auc,0.650610
ccp_probability_min,0.001067


Useful diagnostics:

- monetary-cost R-squared should be reasonably high;
- CCP probabilities must not be concentrated at the clipping bounds;
- continuation R-squared values show whether the one-period transition
  approximation is working;
- the structural fit must converge.

In [5]:
{
    "structural_converged": result_a.structural.converged,
    "sigma": result_a.structural.sigma,
    "calibrated_beta": result_a.structural.beta,
    "ccp_auc": result_a.ccp.diagnostics["roc_auc"],
    "gmm_j": result_a.structural.j_statistic,
}

{'structural_converged': True,
 'sigma': 47.81332478381348,
 'calibrated_beta': 0.96,
 'ccp_auc': 0.6506097012426476,
 'gmm_j': 35092.982417172105}

## 2. Sensitivity to beta

This does not change the official default. It is useful for checking the
calibration. `estimate_beta=True` estimates beta as a nuisance parameter,
but it is weakly identified and may approach its upper bound.

In [6]:
# sensitivity_a = fit_practicum1(
#     DATA / "set_A.parquet",
#     spec="A",
#     config=P1FitConfig(fixed_beta=0.95),
# )
# sensitivity_a.summary()

In [7]:
# weak_id_check = fit_practicum1(
#     DATA / "set_A.parquet",
#     spec="A",
#     config=P1FitConfig(estimate_beta=True),
# )
# weak_id_check.summary()

## 3. Fit A, B, and C

In [8]:
all_results = fit_practicum1_all(DATA, config=config)
all_results.summary()

,dataset,quantity,estimate
0,A,sigma,47.813325
1,A,beta (calibration; not submitted),0.960000
2,A,theta_const,-356.161916
3,A,theta_log_assets,459.366358
4,A,theta_log_assets^2,-20.268342
5,A,theta_npf_a,6821.286014
6,A,theta_roa,-829.263120
7,A,theta_realest_a,4516.696201
8,A,theta_house,0.038824
9,A,theta_house*npf_a,-168.711315


## 4. Provisional counterfactuals

The local instruction does not define the shock magnitude, horizon, or
reported statistic. The following cell must therefore be consciously enabled.
Its assumptions are:

- `no_political`: set `house` and `senate` to zero;
- `myopic`: set beta to zero;
- `npl_stress`: add one sample standard deviation to `npf_a`;
- report the one-period mean closure-probability level.

Replace this block when the organizer supplies the exact definitions.

In [9]:
provisional_cf = P1CounterfactualConfig(
    acknowledge_provisional=True,
    npl_shift_in_standard_deviations=1.0,
    report="level",
)

all_results_with_cf = fit_practicum1_all(
    DATA,
    config=config,
    counterfactual_config=provisional_cf,
)
all_results_with_cf.summary()

/Users/bc3139/repo/summer2026/DSE2026/src/dse_practicum/practicum1.py:1002: RuntimeWarning: Practicum 1 counterfactual definitions are provisional: reporting one-period mean closure probabilities.
  result.counterfactuals = compute_provisional_counterfactuals(
/Users/bc3139/repo/summer2026/DSE2026/src/dse_practicum/practicum1.py:1002: RuntimeWarning: Practicum 1 counterfactual definitions are provisional: reporting one-period mean closure probabilities.
  result.counterfactuals = compute_provisional_counterfactuals(
/Users/bc3139/repo/summer2026/DSE2026/src/dse_practicum/practicum1.py:1002: RuntimeWarning: Practicum 1 counterfactual definitions are provisional: reporting one-period mean closure probabilities.
  result.counterfactuals = compute_provisional_counterfactuals(


,dataset,quantity,estimate
0,A,sigma,47.813325
1,A,beta (calibration; not submitted),0.960000
2,A,theta_const,-356.161916
3,A,theta_log_assets,459.366358
4,A,theta_log_assets^2,-20.268342
5,A,theta_npf_a,6821.286014
6,A,theta_roa,-829.263120
7,A,theta_realest_a,4516.696201
8,A,theta_house,0.038824
9,A,theta_house*npf_a,-168.711315


## 5. Create the submission file

This uses the official template as the schema and refuses missing, extra, or
non-finite rows.

In [10]:
submission = all_results_with_cf.to_submission(
    DATA / "sample_submission.csv",
    ROOT / "practicum" / "practicum1_submission.csv",
)
submission

,Id,Prediction
0,A_sigma,47.813325
1,A_theta_const,-356.161916
2,A_theta_log_assets,459.366358
3,A_theta_log_assets^2,-20.268342
4,A_theta_npf_a,6821.286014
5,A_theta_roa,-829.263120
6,A_theta_realest_a,4516.696201
7,A_theta_house,0.038824
8,A_theta_house*npf_a,-168.711315
9,A_cf_no_political,0.069018
